# Prompt 4 — The Core Terraform Workflow: All CLI Commands
## Terraform Associate (004) Exam Study Guide

**Exam Objective 3**: Understand Terraform's core workflow and be able to describe the purpose, flags, and behavior of every essential CLI command.

---

**Topics covered in this notebook:**
1. The Core Workflow Overview
2. `terraform init` — Initialize a Working Directory
3. `terraform fmt` — Format Configuration Files
4. `terraform validate` — Validate Configuration Syntax
5. `terraform plan` — Create an Execution Plan
6. `terraform apply` — Apply Changes
7. `terraform destroy` — Destroy Infrastructure
8. `terraform output` — Read Output Values
9. `terraform show` — Inspect State or Plans
10. `terraform graph` — Visualize the Dependency Graph
11. `terraform console` — Interactive Expression Evaluation
12. Command Comparison Quick-Reference
13. Exam-Style Questions (3)
14. Real-World Scenarios (5)

---
## 1. The Core Workflow Overview

The standard Terraform workflow follows a repeating cycle:

```
Write          Plan          Apply
  │              │              │
  ▼              ▼              ▼
Edit .tf  →  terraform    →  terraform
files        plan             apply
                │
                ▼
           Review diff
           before applying
```

### Full Sequence Including Setup and Maintenance

```bash
# 1. Initialize (run once per new workspace, or after adding providers/modules)
terraform init

# 2. Write / edit .tf configuration files

# 3. Format — keep consistent style
terraform fmt

# 4. Validate — catch syntax/logic errors before planning
terraform validate

# 5. Plan — preview changes
terraform plan

# 6. Apply — make changes
terraform apply

# 7. Inspect results
terraform output
terraform show

# 8. Tear down when done
terraform destroy
```

### Workflow Principles

| Principle | Details |
|-----------|--------|
| **Plan before apply** | Always review `terraform plan` output before running `apply` |
| **Save plans** | Use `-out=plan.tfplan` to save plans and ensure `apply` executes exactly what was reviewed |
| **Idempotent applies** | Running `terraform apply` on an already-converged config produces no changes |
| **fmt before commit** | Run `terraform fmt` before committing — makes diffs cleaner |
| **validate in CI** | `terraform validate` in every CI pipeline to catch config errors early |

---
## 2. `terraform init` — Initialize a Working Directory

### Purpose

Prepares the working directory for use by downloading providers, modules, and configuring the backend.

### What It Does

```bash
$ terraform init

Initializing the backend...
Initializing provider plugins...
- Finding hashicorp/aws versions matching "~> 5.0"...
- Installing hashicorp/aws v5.40.0...
- Installed hashicorp/aws v5.40.0 (signed by HashiCorp)
Terraform has created a lock file .terraform.lock.hcl
Terraform has been successfully initialized!
```

1. Reads `required_providers` and `required_version` from `.tf` files
2. Downloads provider plugins into `.terraform/providers/`
3. Downloads module source code into `.terraform/modules/`
4. Configures the backend (local, S3, Terraform Cloud, etc.)
5. Creates or updates `.terraform.lock.hcl`

### Key Flags

| Flag | Effect |
|------|--------|
| `-upgrade` | Re-download providers/modules using latest versions matching constraints |
| `-backend=false` | Skip backend initialization (useful for CI syntax checks) |
| `-reconfigure` | Force re-initialize the backend, ignoring any existing configuration |
| `-migrate-state` | Copy state to a newly configured backend |
| `-plugin-dir=PATH` | Use providers from a local filesystem directory (offline/air-gapped) |
| `-input=false` | Disable interactive prompts (required for CI automation) |

### When to Run `terraform init`

```bash
# Run init when:
# 1. First time working with a configuration
terraform init

# 2. Added or changed a provider in required_providers
terraform init

# 3. Added or changed a module source
terraform init

# 4. Changed backend configuration
terraform init -reconfigure

# 5. Want to upgrade providers to newer versions
terraform init -upgrade
```

> **Exam tip**: `terraform init` is **idempotent** — safe to run multiple times. It will not re-download providers that are already installed at the locked version (unless `-upgrade` is used).

---
## 3. `terraform fmt` — Format Configuration Files

### Purpose

Rewrites Terraform configuration files to the canonical style defined by HashiCorp — consistent indentation, alignment of `=` signs, and spacing.

### Basic Usage

```bash
# Format all .tf files in the current directory
terraform fmt

# Format recursively (including subdirectories / modules)
terraform fmt -recursive

# Check formatting without making changes (CI use)
terraform fmt -check

# Show a diff of what would change without making changes
terraform fmt -diff

# Format a specific file
terraform fmt main.tf
```

### Before and After Example

**Before `terraform fmt`:**
```hcl
resource "aws_instance" "web" {
ami           = "ami-0c55b159cbfafe1f0"
    instance_type="t3.micro"
  tags = {
    Name = "web"
    Environment   = "production"
    Team = "platform"
  }
}
```

**After `terraform fmt`:**
```hcl
resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"
  tags = {
    Name        = "web"
    Environment = "production"
    Team        = "platform"
  }
}
```

### Exit Codes for CI

| Exit Code | Meaning |
|-----------|--------|
| `0` | Success — no files needed formatting (with `-check`) |
| `1` | Error — unexpected problem |
| `2` | Files differ — formatting changes needed (with `-check`) |

### CI Usage Pattern

```yaml
# GitHub Actions step — fail PR if formatting is incorrect
- name: Check Terraform Formatting
  run: terraform fmt -check -recursive
  # Exits with code 2 (fail) if any file needs formatting
  # Developer must run `terraform fmt -recursive` locally and commit
```

> **Exam tip**: `terraform fmt` modifies files **in place**. Use `-check` to verify without modifying (returns non-zero if formatting differs). `-recursive` is needed to format modules in subdirectories.

---
## 4. `terraform validate` — Validate Configuration Syntax

### Purpose

Checks the configuration in the current directory for syntax errors and internal consistency — **without accessing any remote services** or modifying state.

### What It Checks

```bash
$ terraform validate
Success! The configuration is valid.

# Or with errors:
$ terraform validate
╷
│ Error: Reference to undeclared input variable
│
│   on main.tf line 12, in resource "aws_instance" "web":
│   12:   instance_type = var.instance_type
│
│ An input variable with the name "instance_type" has not been declared.
╵
```

### What `validate` Catches

| Error Type | Example |
|-----------|--------|
| Syntax errors | Missing `}`, unclosed strings, invalid HCL |
| Undeclared variables | `var.foo` when `foo` has no `variable {}` block |
| Undeclared resources | `aws_vpc.main.id` when `aws_vpc.main` does not exist |
| Invalid attribute names | `instnce_type` instead of `instance_type` |
| Invalid argument types | Passing a string where a number is required |
| Circular dependencies | Resource A references B which references A |

### What `validate` Does NOT Catch

| Limitation | Why |
|-----------|----|
| Invalid AMI IDs | Only caught at apply time — validate does not call AWS API |
| Insufficient IAM permissions | Only caught at apply time |
| Resource quota exceeded | Only caught at apply time |
| Wrong region | Only caught at plan/apply time |

### Requires `terraform init` First

```bash
# validate needs providers downloaded to check attribute schemas
terraform init
terraform validate

# In CI — skip backend init but still validate
terraform init -backend=false
terraform validate
```

### JSON Output for Tooling

```bash
terraform validate -json
# Returns structured JSON with error details
# Useful for parsing in CI pipelines or IDE integrations
```

> **Exam tip**: `validate` checks **configuration correctness** only — it does not contact providers or check whether real-world resources would succeed. It runs locally and offline (after `init`).

---
## 5. `terraform plan` — Create an Execution Plan

### Purpose

Generates and displays a detailed execution plan showing what Terraform will do if `apply` is run — **without making any changes**.

### What `plan` Does

```bash
$ terraform plan

Terraform used the selected providers to generate the following execution plan.
Resource actions are indicated with the following symbols:
  + create
  ~ update in-place
  - destroy
-/+ destroy and then create replacement

Terraform will perform the following actions:

  # aws_instance.web will be created
  + resource "aws_instance" "web" {
      + ami           = "ami-0c55b159cbfafe1f0"
      + id            = (known after apply)
      + instance_type = "t3.micro"
      + private_ip    = (known after apply)
      + public_ip     = (known after apply)
      + tags          = {
          + "Name" = "web"
        }
    }

Plan: 1 to add, 0 to change, 0 to destroy.
```

### Plan Symbols

| Symbol | Meaning |
|--------|--------|
| `+` | Resource will be **created** |
| `-` | Resource will be **destroyed** |
| `~` | Resource will be **updated in-place** (no downtime for most resources) |
| `-/+` | Resource will be **destroyed then recreated** (replacement — may cause downtime) |
| `<=` | Data source will be **read** |

### Key Flags

| Flag | Effect |
|------|--------|
| `-out=FILE` | Save the plan to a file (prevents plan drift between plan and apply) |
| `-var='key=value'` | Set an input variable value |
| `-var-file=FILE` | Load variable values from a `.tfvars` file |
| `-target=ADDR` | Plan only for a specific resource or module |
| `-refresh=false` | Skip API refresh (faster, trusts state) |
| `-refresh-only` | Only show drift — no config changes planned |
| `-destroy` | Generate a destroy plan (what `-destroy` would do) |
| `-compact-warnings` | Show compact warning output |

### Saving Plans for CI/CD

```bash
# In CI: save plan, then apply the exact saved plan
terraform plan -out=plan.tfplan

# In CD: apply exactly what was reviewed — no surprises
terraform apply plan.tfplan

# Why this matters:
# Without -out, apply re-creates the plan at apply time
# If infra changed between plan and apply, you apply a different plan!
```

### `(known after apply)` Values

Some attributes can only be known after the resource is created — for example, the ID assigned by AWS, a dynamically allocated IP, or a computed ARN. Plan shows these as `(known after apply)`.

> **Exam tip**: `terraform plan` never makes changes. It is purely **read-only** from an infrastructure perspective. Always review the plan summary line (`Plan: X to add, Y to change, Z to destroy`) before running apply.

---
## 6. `terraform apply` — Apply Changes

### Purpose

Executes the changes described by the most recent `terraform plan`, creating, updating, or destroying resources to make actual infrastructure match the desired configuration.

### Default Behavior

```bash
$ terraform apply

# Terraform automatically runs a plan first, then asks for confirmation:
Plan: 2 to add, 0 to change, 0 to destroy.

Do you want to perform these actions?
  Terraform will perform the actions described above.
  Only 'yes' will be accepted to approve.

  Enter a value: yes

aws_vpc.main: Creating...
aws_vpc.main: Creation complete after 2s [id=vpc-0a1b2c3d4e5f6]
aws_subnet.public: Creating...
aws_subnet.public: Creation complete after 1s [id=subnet-0b1c2d3e4f5]

Apply complete! Resources: 2 added, 0 changed, 0 destroyed.

Outputs:
vpc_id = "vpc-0a1b2c3d4e5f6"
```

### Applying a Saved Plan

```bash
# Apply a specific saved plan file — skips confirmation prompt
terraform apply plan.tfplan
# No confirmation needed because the plan was already reviewed
```

### Key Flags

| Flag | Effect |
|------|--------|
| `-auto-approve` | Skip the `yes/no` confirmation prompt (required for CI) |
| `-var='key=value'` | Set a variable value |
| `-var-file=FILE` | Load variables from a `.tfvars` file |
| `-target=ADDR` | Apply only for a specific resource (use sparingly — causes state inconsistency) |
| `-refresh=false` | Skip state refresh before applying |
| `-parallelism=N` | Control how many operations run concurrently (default: 10) |
| `-replace=ADDR` | Force replacement of a specific resource (replaces deprecated `terraform taint`) |

### Apply in CI/CD

```bash
# CI pipeline pattern — non-interactive
terraform apply -auto-approve plan.tfplan

# Or with variables
terraform apply \
  -auto-approve \
  -var-file=production.tfvars
```

### Post-Apply State Update

After a successful apply:
1. Terraform writes the new state to `terraform.tfstate`
2. The previous state is backed up to `terraform.tfstate.backup`
3. Output values are printed to the terminal

### Force-Replace a Resource

```bash
# Replace (destroy + recreate) a specific resource
# Useful when a resource is in a broken state
terraform apply -replace=aws_instance.web

# Equivalent to the old (deprecated) terraform taint command
```

> **Exam tip**: `terraform apply` with a saved plan file (e.g., `terraform apply plan.tfplan`) does **not** prompt for confirmation — the confirmation was implicit when the plan was reviewed. This is the recommended CI/CD pattern.

---
## 7. `terraform destroy` — Destroy Infrastructure

### Purpose

Destroys all infrastructure managed by the current Terraform configuration. Equivalent to removing all resources from `.tf` files and running `apply`.

### Basic Usage

```bash
$ terraform destroy

# Terraform generates a destroy plan first:
Plan: 0 to add, 0 to change, 5 to destroy.

Do you really want to destroy all resources?
  Terraform will destroy all your managed infrastructure, as shown above.
  There is no undo. Only 'yes' will be accepted to confirm.

  Enter a value: yes

aws_instance.web: Destroying... [id=i-0abc123]
aws_instance.web: Destruction complete after 32s
aws_subnet.public: Destroying...
...
Destroy complete! Resources: 5 destroyed.
```

### Equivalent Commands

```bash
# These are equivalent:
terraform destroy
terraform apply -destroy

# Preview what would be destroyed (no changes made)
terraform plan -destroy
```

### Key Flags

| Flag | Effect |
|------|--------|
| `-auto-approve` | Skip confirmation (dangerous — use only in known-safe automation) |
| `-target=ADDR` | Destroy only a specific resource |
| `-var-file=FILE` | Load variables (needed if destroy requires variable values) |

### Destruction Order

Terraform destroys resources in **reverse dependency order** — dependent resources are destroyed before their dependencies:

```
Creation order:     VPC → Subnet → Security Group → EC2 Instance
Destruction order:  EC2 Instance → Security Group → Subnet → VPC
```

### Preventing Accidental Destruction

```hcl
# lifecycle block — prevent accidental destroy of critical resources
resource "aws_db_instance" "production" {
  # ... configuration ...

  lifecycle {
    prevent_destroy = true
    # Terraform will error if you try to destroy this resource
    # Must set prevent_destroy = false first before destroying
  }
}
```

> **Exam tip**: `terraform destroy` is **irreversible** for most resources. Always run `terraform plan -destroy` first to see exactly what will be removed. `prevent_destroy = true` in the `lifecycle` block is the safety net for production databases and stateful resources.

---
## 8. `terraform output` — Read Output Values

### Purpose

Reads and displays the values of `output {}` blocks defined in the configuration — from the current state file. Useful for sharing values between Terraform workspaces or with external scripts.

### Output Block Definition

```hcl
# Define outputs in outputs.tf
output "vpc_id" {
  value       = aws_vpc.main.id
  description = "The ID of the main VPC"
}

output "db_endpoint" {
  value       = aws_db_instance.main.endpoint
  description = "RDS connection endpoint"
  sensitive   = true  # Redacts value in terminal output
}

output "instance_ips" {
  value = [for i in aws_instance.web : i.public_ip]
}
```

### Usage

```bash
# Show all outputs
$ terraform output
db_endpoint = <sensitive>
instance_ips = [
  "54.210.155.204",
  "54.210.155.205",
]
vpc_id = "vpc-0a1b2c3d4e5f6"

# Show a specific output
$ terraform output vpc_id
"vpc-0a1b2c3d4e5f6"

# Show raw value (no quotes) — for use in shell scripts
$ terraform output -raw vpc_id
vpc-0a1b2c3d4e5f6

# Show all outputs as JSON (for scripting / pipeline integration)
$ terraform output -json
{
  "vpc_id": { "value": "vpc-0a1b2c3d4e5f6", "type": "string", "sensitive": false },
  "db_endpoint": { "value": "prod-db.abc.us-east-1.rds.amazonaws.com:5432", "type": "string", "sensitive": true }
}

# Show sensitive value
$ terraform output -raw db_endpoint
prod-db.abc.us-east-1.rds.amazonaws.com:5432
# Note: -raw and -json bypass the sensitive redaction in terminal output
```

### Consuming Outputs in Shell Scripts

```bash
# Get VPC ID for use in another script
VPC_ID=$(terraform output -raw vpc_id)
echo "Deploying into VPC: $VPC_ID"

# Get JSON output for parsing with jq
terraform output -json | jq '.instance_ips.value[]'
# "54.210.155.204"
# "54.210.155.205"
```

### Remote State Data Source (Cross-Workspace Outputs)

```hcl
# Consume outputs from another Terraform workspace
data "terraform_remote_state" "networking" {
  backend = "s3"
  config = {
    bucket = "company-terraform-state"
    key    = "networking/terraform.tfstate"
    region = "us-east-1"
  }
}

resource "aws_instance" "app" {
  subnet_id = data.terraform_remote_state.networking.outputs.public_subnet_id
  # Uses vpc_id output from the networking workspace
}
```

> **Exam tip**: `terraform output -raw` returns a value without quotes (plain string) — ideal for shell scripts. `terraform output -json` returns all outputs as a JSON object — ideal for CI tooling. Sensitive outputs are redacted in plain `terraform output` but revealed by `-raw` and `-json`.

---
## 9. `terraform show` — Inspect State or Plans

### Purpose

Renders the current state file **or** a saved plan file in human-readable format.

### Usage

```bash
# Show current state (all managed resources)
terraform show

# Show current state as JSON
terraform show -json

# Show a saved plan file
terraform show plan.tfplan

# Show a saved plan as JSON (for CI policy tools like OPA/Sentinel)
terraform show -json plan.tfplan
```

### When to Use `terraform show` vs `terraform state show`

| Command | Input | Shows |
|---------|-------|-------|
| `terraform show` | Entire state | All resources formatted as HCL |
| `terraform show -json` | Entire state | All resources as JSON |
| `terraform show plan.tfplan` | Saved plan file | What the plan will do |
| `terraform state show <addr>` | State + address | One specific resource's attributes |
| `terraform state list` | State | List of resource addresses only |

### Plan File Inspection Pattern

```bash
# 1. Generate plan
terraform plan -out=plan.tfplan

# 2. Inspect the plan — human readable
terraform show plan.tfplan

# 3. Policy check with OPA (Open Policy Agent)
terraform show -json plan.tfplan > plan.json
opa eval -d policy.rego -i plan.json 'data.terraform.allow'

# 4. If policy passes, apply the exact plan
terraform apply plan.tfplan
```

> **Exam tip**: A saved plan file (`.tfplan`) is **binary** — it cannot be read with a text editor. Use `terraform show plan.tfplan` to view it, and `terraform show -json plan.tfplan` for machine-readable output.

---
## 10. `terraform graph` — Visualize the Dependency Graph

### Purpose

Outputs a visual representation of the dependency graph between all resources in the configuration, in DOT format (Graphviz).

### Usage

```bash
# Generate the graph in DOT format
terraform graph

# Pipe to Graphviz to generate a PNG image
terraform graph | dot -Tpng > graph.png

# Generate graph for a specific operation type
terraform graph -type=plan
terraform graph -type=apply
terraform graph -type=destroy

# Save DOT file and view in browser
terraform graph > graph.dot
# Paste contents into: https://dreampuf.github.io/GraphvizOnline/
```

### Example DOT Output

```dot
digraph {
  compound = "true"
  newrank = "true"
  subgraph "root" {
    "[root] aws_subnet.public" [label = "aws_subnet.public", shape = "box"]
    "[root] aws_vpc.main" [label = "aws_vpc.main", shape = "box"]
    "[root] aws_instance.web" [label = "aws_instance.web", shape = "box"]
    "[root] aws_subnet.public" -> "[root] aws_vpc.main"
    "[root] aws_instance.web" -> "[root] aws_subnet.public"
  }
}
```

### When to Use `terraform graph`

| Use Case | Description |
|----------|-------------|
| **Debugging dependency issues** | Understand why resources are created in a specific order |
| **Documentation** | Generate visual architecture diagrams from IaC |
| **Cycle detection** | Find circular dependencies (Terraform will error on cycles) |
| **Complex module analysis** | Understand how modules depend on each other |

> **Exam tip**: `terraform graph` outputs **DOT format** (Graphviz). It requires an external tool (`dot`) to render as an image. The graph shows the **dependency order** Terraform uses when creating and destroying resources.

---
## 11. `terraform console` — Interactive Expression Evaluation

### Purpose

Opens an interactive REPL (Read-Eval-Print Loop) for evaluating Terraform expressions against the current state and configuration.

### Usage

```bash
$ terraform console
> 
```

### What You Can Do in the Console

```hcl
# Evaluate expressions against state
> aws_vpc.main.id
"vpc-0a1b2c3d4e5f6"

# Test built-in functions
> max(5, 12, 9)
12

> length(["a", "b", "c"])
3

> upper("hello")
"HELLO"

> toset(["a", "b", "a"])
toset([
  "a",
  "b",
])

# Test complex expressions
> [for ip in aws_instance.web : ip.public_ip]
[
  "54.210.155.204",
  "54.210.155.205",
]

# Test string interpolation
> "The VPC is ${aws_vpc.main.id}"
"The VPC is vpc-0a1b2c3d4e5f6"

# Inspect variables
> var.environment
"production"

# Test cidrsubnet function
> cidrsubnet("10.0.0.0/16", 8, 1)
"10.0.1.0/24"

> cidrhost("10.0.1.0/24", 5)
"10.0.1.5"

# Exit console
> exit
# or Ctrl+D
```

### Non-Interactive (Scripted) Usage

```bash
# Evaluate a single expression non-interactively
echo 'aws_vpc.main.id' | terraform console

# Useful in shell scripts to extract state values without parsing JSON
VPC_ID=$(echo 'aws_vpc.main.id' | terraform console | tr -d '"')
```

### When to Use `terraform console`

| Use Case | Example |
|----------|--------|
| **Test functions** | Try `cidrsubnet`, `jsonencode`, `flatten` before using in config |
| **Debug expressions** | Check what a complex `for` expression produces |
| **Inspect state values** | Read resource attributes without editing config |
| **Validate variable types** | Confirm a variable has the expected type and value |

> **Exam tip**: `terraform console` has **read-only access** to state — it cannot make changes. It is a debugging and development tool. Requires `terraform init` to have been run.

---
## 12. Command Comparison Quick-Reference

### All Core Commands at a Glance

| Command | Changes Infrastructure? | Changes State? | Requires Init? | Contacts Provider API? |
|---------|------------------------|----------------|----------------|------------------------|
| `terraform init` | No | No (lock file only) | N/A | Yes (downloads providers) |
| `terraform fmt` | No | No | No | No |
| `terraform validate` | No | No | Yes (needs provider schemas) | No |
| `terraform plan` | No | No | Yes | Yes (refresh by default) |
| `terraform apply` | **Yes** | **Yes** | Yes | **Yes** |
| `terraform destroy` | **Yes** | **Yes** | Yes | **Yes** |
| `terraform output` | No | No | Yes | No |
| `terraform show` | No | No | Yes | No |
| `terraform graph` | No | No | Yes | No |
| `terraform console` | No | No | Yes | No (reads state) |

### Workflow Stage Mapping

| Stage | Commands | Purpose |
|-------|---------|--------|
| Setup | `init` | Download providers, configure backend |
| Write | Edit `.tf` files | Define desired infrastructure |
| Verify | `fmt`, `validate` | Ensure config is correct and formatted |
| Plan | `plan` | Preview changes safely |
| Apply | `apply` | Execute changes |
| Inspect | `output`, `show`, `state show`, `console` | Read results and values |
| Debug | `graph`, `console`, `state list` | Understand structure and dependencies |
| Teardown | `destroy` | Remove all managed infrastructure |

### Common Flag Patterns

```bash
# CI/CD pipeline pattern
terraform init -input=false
terraform validate
terraform plan -out=plan.tfplan -input=false
terraform apply -auto-approve plan.tfplan

# Local development pattern
terraform init
terraform fmt -recursive
terraform validate
terraform plan
terraform apply

# Safe teardown of dev environment
terraform plan -destroy
terraform destroy -auto-approve  # Only after reviewing plan!
```

---
## 13. Exam-Style Practice Questions

---

**Q1: A CI pipeline runs `terraform plan -out=plan.tfplan` in one step and `terraform apply plan.tfplan` in a later step. Why is saving the plan to a file a best practice in CI/CD?**

A) The `-out` flag compresses the plan to reduce storage space  
B) Saving the plan to a file ensures that the `apply` step executes exactly the same changes that were reviewed in the plan step — even if infrastructure changed between the two steps  
C) It is required when using remote backends — apply will fail without a plan file  
D) The plan file encrypts sensitive variable values before they are passed to apply  

<details>
<summary>Answer</summary>

**Answer: B**  
Without `-out`, `terraform apply` generates a **new** plan at apply time. If infrastructure changed between the plan and apply steps (another team member applied, or an external change occurred), you could apply a different plan than what was reviewed. Saving to a file with `-out` guarantees apply executes the exact same plan. Option A, C, and D are all incorrect.

</details>

---

**Q2: What is the difference between `terraform validate` and `terraform plan` in terms of what they check?**

A) `validate` checks the state file; `plan` checks the configuration files  
B) `validate` checks configuration syntax and internal consistency without contacting providers; `plan` contacts providers to compute a diff against actual infrastructure  
C) `validate` and `plan` perform identical checks — `validate` is just a faster alias for `plan`  
D) `validate` requires AWS credentials; `plan` can run without credentials  

<details>
<summary>Answer</summary>

**Answer: B**  
`terraform validate` checks HCL syntax, variable references, and attribute names — all locally, without API calls. It catches typos and logical errors in the configuration. `terraform plan` actually contacts provider APIs (to refresh state), computes what changes are needed, and shows a diff. `plan` catches things `validate` cannot — like an invalid AMI ID or an expired credential.

</details>

---

**Q3: An engineer wants to test the Terraform built-in function `cidrsubnet("192.168.0.0/16", 8, 3)` before using it in a configuration. Which command should they use?**

A) `terraform validate -test`  
B) `terraform plan -test-expr`  
C) `terraform console`  
D) `terraform show -expr`  

<details>
<summary>Answer</summary>

**Answer: C**  
`terraform console` is the interactive REPL for evaluating Terraform expressions and testing built-in functions. The engineer types the function expression at the `>` prompt and immediately sees the result. Options A, B, and D are not valid Terraform commands or flags.

</details>

---
## 14. Real-World Scenarios

<details>
<summary>Scenario 1 — Full CI/CD Pipeline: Plan on PR, Apply on Merge</summary>

**Situation:**
A platform engineering team wants automated Terraform workflows: every pull request shows a plan as a PR comment, and merges to main trigger an automated apply.

**GitHub Actions workflow:**

```yaml
# .github/workflows/terraform.yml
name: Terraform

on:
  pull_request:
    branches: [main]
  push:
    branches: [main]

jobs:
  terraform:
    runs-on: ubuntu-latest
    defaults:
      run:
        working-directory: infrastructure/

    steps:
      - uses: actions/checkout@v4

      - name: Setup Terraform
        uses: hashicorp/setup-terraform@v3

      - name: Terraform Init
        run: terraform init -input=false
        env:
          AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
          AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}

      - name: Terraform Format Check
        run: terraform fmt -check -recursive

      - name: Terraform Validate
        run: terraform validate

      - name: Terraform Plan
        run: terraform plan -out=plan.tfplan -input=false
        # Plan output is added to PR as a comment

      - name: Terraform Apply
        if: github.ref == 'refs/heads/main' && github.event_name == 'push'
        run: terraform apply -auto-approve plan.tfplan
        # Only runs on merge to main — not on PRs
```

**Expected outcome:**
- PRs show a plan — developers can see infrastructure changes before merging
- Format and validate catch errors early
- Apply uses the exact saved plan — no surprises on merge
- Credentials stored as GitHub secrets — never in code

**Exam sub-objective demonstrated:** Full workflow (`init` → `fmt` → `validate` → `plan` → `apply`), saved plan files, `-auto-approve`, `-input=false` for CI.

</details>

<details>
<summary>Scenario 2 — Debugging a `terraform plan` Replacement Warning</summary>

**Situation:**
A developer changes the `ami` attribute of an EC2 instance in the configuration. The `terraform plan` output shows `-/+` (destroy then recreate). They are surprised — they expected an in-place update. They need to understand why and decide whether to proceed.

**Reading the plan carefully:**

```bash
$ terraform plan

  # aws_instance.web must be replaced
-/+ resource "aws_instance" "web" {
      ~ id            = "i-0abc123" -> (known after apply)
      ~ private_ip    = "10.0.1.45" -> (known after apply)
      ~ public_ip     = "54.210.155.204" -> (known after apply)
      -/+ ami         = "ami-old" -> "ami-new" (forces replacement)
        # instance_type = "t3.micro"  (unchanged)
    }

Plan: 1 to add, 0 to change, 1 to destroy.
```

**Analysis using `terraform graph`:**

```bash
# Check what depends on this instance
terraform graph | grep 'aws_instance.web'
# Shows: aws_eip.web -> aws_instance.web
# This means the Elastic IP is associated with this instance
# Destroying the instance will also affect the EIP association

# Full plan review
terraform plan -out=ami-update.tfplan
terraform show ami-update.tfplan | grep -A 20 'aws_eip'
# Confirms EIP association will be updated automatically
```

**Decision options:**

```bash
# Option 1: Accept replacement (EC2 AMI changes always require replacement)
terraform apply ami-update.tfplan
# Brief downtime while old instance is destroyed and new one starts

# Option 2: Use a launch template with rolling replacement
# For zero-downtime, use Auto Scaling Group with rolling update policy
# instead of a standalone aws_instance resource
```

**Expected outcome:**
- Developer understands that `ami` is an **immutable attribute** for EC2
- Replacement is expected and intended behavior
- `terraform graph` confirms downstream impact

**Exam sub-objective demonstrated:** Plan symbols (`-/+`), immutable vs mutable attributes, `terraform graph` for dependency analysis.

</details>

<details>
<summary>Scenario 3 — Using `terraform console` to Debug a Complex Expression</summary>

**Situation:**
An engineer is writing a Terraform configuration that uses `cidrsubnet` to generate subnet CIDRs dynamically for 4 availability zones. Before hardcoding the wrong values, they want to test the function interactively.

**Interactive console session:**

```bash
$ terraform console

# Test cidrsubnet to understand the output
> cidrsubnet("10.0.0.0/16", 8, 0)
"10.0.0.0/24"

> cidrsubnet("10.0.0.0/16", 8, 1)
"10.0.1.0/24"

> cidrsubnet("10.0.0.0/16", 8, 2)
"10.0.2.0/24"

> cidrsubnet("10.0.0.0/16", 8, 3)
"10.0.3.0/24"

# Test the for expression that will be used in config
> [for i in range(4) : cidrsubnet("10.0.0.0/16", 8, i)]
[
  "10.0.0.0/24",
  "10.0.1.0/24",
  "10.0.2.0/24",
  "10.0.3.0/24",
]

# Test with a map for for_each
> { for i, az in ["us-east-1a", "us-east-1b", "us-east-1c"] : az => cidrsubnet("10.0.0.0/16", 8, i) }
{
  "us-east-1a" = "10.0.0.0/24"
  "us-east-1b" = "10.0.1.0/24"
  "us-east-1c" = "10.0.2.0/24"
}

> exit
```

**Now confident, the engineer writes the config:**

```hcl
variable "availability_zones" {
  default = ["us-east-1a", "us-east-1b", "us-east-1c"]
}

resource "aws_subnet" "public" {
  for_each = { for i, az in var.availability_zones : az => cidrsubnet("10.0.0.0/16", 8, i) }

  vpc_id            = aws_vpc.main.id
  availability_zone = each.key
  cidr_block        = each.value
}
```

**Expected outcome:**
- Zero trial-and-error in the actual config — expression tested and verified first
- Correct subnet CIDRs allocated per AZ
- `terraform validate` passes immediately

**Exam sub-objective demonstrated:** `terraform console` for function testing, REPL-driven development, `cidrsubnet` and `for` expressions.

</details>

<details>
<summary>Scenario 4 — Targeted Apply for Faster Iteration During Development</summary>

**Situation:**
A developer is working on a new Lambda function configuration. The workspace also contains 200 other resources (VPCs, RDS instances, EKS clusters). Every full `terraform apply` takes 8 minutes due to the refresh step and plan computation. They want fast iteration on just the Lambda function.

**Using `-target` for targeted operations:**

```bash
# Fast iteration: only plan/apply the Lambda function
terraform plan -target=aws_lambda_function.processor
terraform apply -target=aws_lambda_function.processor
# Completes in ~15 seconds instead of 8 minutes

# Target an entire module
terraform apply -target=module.lambda_functions

# After development is done — run a full apply to ensure consistency
terraform plan
terraform apply
# Full apply with no targets ensures no state drift from targeted ops
```

**Important caveat:**

```
WARNING: Using -target can lead to inconsistent state.

Terraform shows this warning:
"Warning: Applied changes may be incomplete
The plan was created with the -target option in effect, so some changes
requested in the configuration may have been ignored and the output values
may not be fully updated. Run the following command to verify that the plan
is complete:
    terraform plan"
```

**Expected outcome:**
- Fast development iteration — seconds instead of minutes
- Full apply run at end of development session to ensure all resources are consistent
- `-target` treated as a development-only tool, never used in production CI

**Exam sub-objective demonstrated:** `terraform plan -target`, `terraform apply -target`, state consistency risks of targeted operations.

</details>

<details>
<summary>Scenario 5 — `terraform fmt` as a Pre-Commit Hook</summary>

**Situation:**
A team's CI pipeline fails on `terraform fmt -check` because different team members use different editors with inconsistent Terraform formatting settings. Some use 2-space indent, others use tabs, and `=` alignment is inconsistent. The CI check blocks every PR until formatting is fixed manually.

**Solution: git pre-commit hook:**

```bash
# Option 1: Manual pre-commit hook
cat > .git/hooks/pre-commit << 'EOF'
#!/bin/sh
# Auto-format Terraform files before every commit
terraform fmt -recursive

# Stage the formatted files
git diff --name-only --diff-filter=M | grep '.tf$' | xargs git add
EOF
chmod +x .git/hooks/pre-commit
```

```yaml
# Option 2: pre-commit framework (.pre-commit-config.yaml)
repos:
  - repo: https://github.com/antonbabenko/pre-commit-terraform
    rev: v1.92.0
    hooks:
      - id: terraform_fmt        # Auto-formats on every commit
      - id: terraform_validate   # Validates syntax on every commit
      - id: terraform_docs       # Updates README with module docs
```

```bash
# Install and enable
pip install pre-commit
pre-commit install

# Now every git commit automatically runs:
# 1. terraform fmt -recursive   (formats .tf files)
# 2. terraform validate         (catches syntax errors)
# 3. terraform_docs             (updates module documentation)
```

**Expected outcome:**
- Every committed `.tf` file is correctly formatted automatically
- CI `fmt -check` always passes — no more formatting-related PR failures
- `validate` errors caught before push — faster feedback loop
- Team-wide consistency without requiring editor configuration standardization

**Exam sub-objective demonstrated:** `terraform fmt -check`, `terraform fmt -recursive`, `terraform validate` in the write-plan-apply workflow.

</details>

---
## Quick-Reference Cheat Sheet

```
Core Terraform CLI Commands Cheat Sheet
═══════════════════════════════════════════════════════════════════

SETUP
  terraform init                   Download providers + modules, configure backend
  terraform init -upgrade          Upgrade to newer matching provider versions
  terraform init -backend=false    Skip backend (CI syntax check)
  terraform init -reconfigure      Force re-init backend

WRITE PHASE
  terraform fmt                    Format current directory .tf files in-place
  terraform fmt -recursive         Format all subdirectories too
  terraform fmt -check             Check only (non-zero exit if changes needed)
  terraform validate               Syntax + reference check (no API calls)
  terraform validate -json         Machine-readable output

PLAN PHASE
  terraform plan                   Preview changes (no changes applied)
  terraform plan -out=FILE         Save plan to file (use for CI/CD)
  terraform plan -destroy          Preview destroy
  terraform plan -target=ADDR      Plan only specific resource
  terraform plan -refresh=false    Skip API refresh (faster)
  terraform plan -refresh-only     Only show drift, no config changes

  Plan symbols:
    +    create      -    destroy
    ~    update      -/+  destroy+recreate (replacement)

APPLY PHASE
  terraform apply                  Apply with confirmation prompt
  terraform apply plan.tfplan      Apply saved plan (no prompt)
  terraform apply -auto-approve    Skip confirmation (CI)
  terraform apply -replace=ADDR    Force recreate a resource
  terraform apply -destroy         Destroy all resources
  terraform destroy                Same as apply -destroy

INSPECT
  terraform output                 Show all outputs
  terraform output -raw NAME       Raw value (no quotes, for shell)
  terraform output -json           All outputs as JSON
  terraform show                   Show entire state as HCL
  terraform show plan.tfplan       Show saved plan
  terraform show -json             Show state/plan as JSON
  terraform state list             List all resource addresses
  terraform state show ADDR        Show one resource's attributes

DEBUG
  terraform graph                  Dependency graph (DOT format)
  terraform graph | dot -Tpng > g  Render as PNG image
  terraform console                Interactive expression REPL

CI/CD PATTERN
  terraform init -input=false
  terraform fmt -check -recursive
  terraform validate
  terraform plan -out=plan.tfplan -input=false
  terraform apply -auto-approve plan.tfplan
═══════════════════════════════════════════════════════════════════
```